# HDB Resale Flat Prices ETL Pipeline

## Overview
This notebook implements an ETL (Extract, Transform, Load) pipeline for HDB resale flat prices data from **January 2012 to December 2016**.

## Pipeline Stages
1. **Extraction** - Load and combine CSV files covering the target date range
2. **Data Profiling** - Statistical analysis and quality assessment of the master dataset
3. **Data Validation & Cleaning** - Apply validation rules and clean the data
4. **Data Transformation** - Create derived columns (Resale Identifier, Hashed ID)
5. **Output** - Export Raw, Cleaned, Transformed, Failed, and Hashed datasets

## Data Sources
Three CSV files contribute records within the Jan 2012 - Dec 2016 window:
- `Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv` (Jan 2012 - Feb 2012)
- `Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv`
- `Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv`

## Schema Notes
- The 2000-Feb 2012 file has 10 columns (no `remaining_lease`)
- The Mar 2012-Dec 2014 file has 10 columns (no `remaining_lease`)
- The Jan 2015-Dec 2016 file has 10 columns (no `remaining_lease`)
- The `remaining_lease` column will be recomputed from `lease_commence_date` assuming a 99-year lease

## Anomaly Detection Heuristics
The pipeline uses the following heuristics to identify potentially anomalous resale prices:

1. **IQR Method by Group**: For each combination of (town, flat_type, storey_range), compute the Interquartile Range (IQR). Prices below Q1 - 1.5*IQR or above Q3 + 1.5*IQR are flagged as anomalous. This approach accounts for the fact that resale prices vary significantly by location, flat type, and floor level.

2. **Price-per-SQM Z-Score**: Compute price per square metre, then calculate z-scores within each (town, flat_type) group. Records with |z-score| > 3 are flagged. This normalises for unit size differences.

## Hashing Algorithm
The pipeline uses **SHA-256** for hashing the Resale Identifier:
- SHA-256 is a cryptographic hash function from the SHA-2 family
- It produces a 256-bit (64 hex character) digest
- It is **irreversible** (one-way function) - the original identifier cannot be recovered from the hash
- It preserves **uniqueness** - different inputs produce different outputs (collision-resistant)
- It is deterministic - the same input always produces the same hash
- It is widely adopted and considered cryptographically secure

## 0. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import hashlib
import os
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Define paths
BASE_DIR = Path('.')
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# Date range filter
START_DATE = '2012-01'
END_DATE = '2016-12'

# HDB lease assumption
HDB_LEASE_YEARS = 99

# Today's date for remaining lease computation
TODAY = datetime.today()

print(f"Pipeline execution date: {TODAY.strftime('%Y-%m-%d')}")
print(f"Target date range: {START_DATE} to {END_DATE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Pipeline execution date: 2026-03-09
Target date range: 2012-01 to 2016-12
Output directory: C:\Users\Kenny\Downloads\ResaleFlatPrices\output


---
## 1. Data Extraction
Load all CSV files that contain records within the Jan 2012 - Dec 2016 range, filter to the target period, and combine into a single master dataset.

In [2]:
# Define source files that may contain records in our target range
SOURCE_FILES = [
    'Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv',
    'Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv',
    'Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv',
]

dfs_raw = []
for file_name in SOURCE_FILES:
    file_path = BASE_DIR / file_name
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['_source_file'] = file_name  # Track source for auditing
        print(f"Loaded: {file_name} -> {len(df)} rows, {list(df.columns)}")
        dfs_raw.append(df)
    else:
        print(f"WARNING: File not found: {file_name}")

# Combine all raw data (union with alignment on column names)
df_all_raw = pd.concat(dfs_raw, ignore_index=True, sort=False)
print(f"\nTotal raw records loaded: {len(df_all_raw)}")
print(f"Columns: {list(df_all_raw.columns)}")

Loaded: Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv -> 369651 rows, ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', '_source_file']
Loaded: Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv -> 52203 rows, ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', '_source_file']
Loaded: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv -> 37153 rows, ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', '_source_file']

Total raw records loaded: 459007
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', '_source_file', 'remaining_le

In [3]:
# Filter to the target date range: Jan 2012 to Dec 2016
df_all_raw['month'] = df_all_raw['month'].astype(str).str.strip()
df_master = df_all_raw[
    (df_all_raw['month'] >= START_DATE) & (df_all_raw['month'] <= END_DATE)
].copy()

print(f"Records within {START_DATE} to {END_DATE}: {len(df_master)}")
print(f"Date range in filtered data: {df_master['month'].min()} to {df_master['month'].max()}")
print(f"\nRecords by source file:")
print(df_master['_source_file'].value_counts())

Records within 2012-01 to 2016-12: 92544
Date range in filtered data: 2012-01 to 2016-12

Records by source file:
_source_file
Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv    52203
Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv    37153
Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv                   3188
Name: count, dtype: int64


In [4]:
# Save raw output (as-is, before any cleaning or transformation)
raw_output_path = OUTPUT_DIR / 'raw_data.csv'
df_master.drop(columns=['_source_file']).to_csv(raw_output_path, index=False)
print(f"Raw data saved to: {raw_output_path} ({len(df_master)} rows)")
df_master.head()

Raw data saved to: output\raw_data.csv (92544 rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,_source_file,remaining_lease
366463,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN
366464,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN
366465,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN
366466,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN
366467,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN


---
## 2. Data Profiling
Perform comprehensive data profiling to understand the statistical properties of the master dataset.

In [5]:
# Basic dataset info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df_master.shape}")
print(f"Memory usage: {df_master.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print()
df_master.info()

DATASET OVERVIEW
Shape: (92544, 12)
Memory usage: 49.31 MB

<class 'pandas.DataFrame'>
RangeIndex: 92544 entries, 366463 to 459006
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                92544 non-null  str    
 1   town                 92544 non-null  str    
 2   flat_type            92544 non-null  str    
 3   block                92544 non-null  str    
 4   street_name          92544 non-null  str    
 5   storey_range         92544 non-null  str    
 6   floor_area_sqm       92544 non-null  float64
 7   flat_model           92544 non-null  str    
 8   lease_commence_date  92544 non-null  int64  
 9   resale_price         92544 non-null  float64
 10  _source_file         92544 non-null  str    
 11  remaining_lease      37153 non-null  float64
dtypes: float64(3), int64(1), str(8)
memory usage: 8.5 MB


In [6]:
# Data type and null analysis
print("=" * 60)
print("NULL VALUES AND DATA TYPES")
print("=" * 60)
profiling_summary = pd.DataFrame({
    'dtype': df_master.dtypes,
    'non_null_count': df_master.count(),
    'null_count': df_master.isnull().sum(),
    'null_pct': (df_master.isnull().sum() / len(df_master) * 100).round(2),
    'unique_count': df_master.nunique(),
    'sample_value': df_master.iloc[0]
})
print(profiling_summary)

NULL VALUES AND DATA TYPES
                       dtype  non_null_count  null_count  null_pct  \
month                    str           92544           0      0.00   
town                     str           92544           0      0.00   
flat_type                str           92544           0      0.00   
block                    str           92544           0      0.00   
street_name              str           92544           0      0.00   
storey_range             str           92544           0      0.00   
floor_area_sqm       float64           92544           0      0.00   
flat_model               str           92544           0      0.00   
lease_commence_date    int64           92544           0      0.00   
resale_price         float64           92544           0      0.00   
_source_file             str           92544           0      0.00   
remaining_lease      float64           37153       55391     59.85   

                     unique_count  \
month                    

In [7]:
# Statistical summary for numeric columns
print("=" * 60)
print("NUMERIC COLUMN STATISTICS")
print("=" * 60)
print(df_master.describe())

NUMERIC COLUMN STATISTICS
       floor_area_sqm  lease_commence_date  resale_price  remaining_lease
count    92544.000000         92544.000000  9.254400e+04     37153.000000
mean        96.569115          1990.072701  4.509390e+05        73.913116
std         24.682292            10.446719  1.281813e+05        10.885456
min         31.000000          1966.000000  1.900000e+05        48.000000
25%         74.000000          1983.000000  3.570000e+05        66.000000
50%         95.000000          1988.000000  4.280000e+05        72.000000
75%        111.000000          1999.000000  5.150000e+05        83.000000
max        280.000000          2013.000000  1.150000e+06        97.000000


In [8]:
# Categorical column value distributions
print("=" * 60)
print("CATEGORICAL COLUMN DISTRIBUTIONS")
print("=" * 60)

categorical_cols = ['town', 'flat_type', 'flat_model', 'storey_range']
for col in categorical_cols:
    print(f"\n--- {col} ({df_master[col].nunique()} unique values) ---")
    print(df_master[col].value_counts())

CATEGORICAL COLUMN DISTRIBUTIONS

--- town (26 unique values) ---
town
JURONG WEST        7573
WOODLANDS          7399
TAMPINES           6728
BEDOK              6071
YISHUN             5937
SENGKANG           5896
HOUGANG            4735
ANG MO KIO         4558
CHOA CHU KANG      3928
BUKIT BATOK        3773
BUKIT MERAH        3618
BUKIT PANJANG      3261
PASIR RIS          3075
PUNGGOL            3075
TOA PAYOH          2917
KALLANG/WHAMPOA    2755
GEYLANG            2649
QUEENSTOWN         2524
CLEMENTI           2319
SEMBAWANG          2314
JURONG EAST        2157
SERANGOON          2029
BISHAN             1671
CENTRAL AREA        750
MARINE PARADE       640
BUKIT TIMAH         192
Name: count, dtype: int64

--- flat_type (7 unique values) ---
flat_type
4 ROOM              36535
3 ROOM              26307
5 ROOM              21368
EXECUTIVE            7295
2 ROOM                956
1 ROOM                 56
MULTI-GENERATION       27
Name: count, dtype: int64

--- flat_model (20 uniq


--- storey_range (25 unique values) ---
storey_range
04 TO 06    21214
07 TO 09    18765
01 TO 03    17466
10 TO 12    16028
13 TO 15     6704
16 TO 18     2706
01 TO 05     2700
06 TO 10     2474
11 TO 15     1259
19 TO 21     1156
22 TO 24      746
25 TO 27      381
16 TO 20      265
28 TO 30      236
21 TO 25       92
34 TO 36       88
37 TO 39       81
31 TO 33       79
26 TO 30       39
40 TO 42       38
46 TO 48        8
43 TO 45        8
36 TO 40        7
31 TO 35        2
49 TO 51        2
Name: count, dtype: int64


In [9]:
# Date distribution
print("=" * 60)
print("MONTHLY RECORD DISTRIBUTION")
print("=" * 60)
monthly_counts = df_master['month'].value_counts().sort_index()
print(monthly_counts)
print(f"\nMonths covered: {len(monthly_counts)}")
print(f"Avg records per month: {monthly_counts.mean():.0f}")
print(f"Min records in a month: {monthly_counts.min()} ({monthly_counts.idxmin()})")
print(f"Max records in a month: {monthly_counts.max()} ({monthly_counts.idxmax()})")

MONTHLY RECORD DISTRIBUTION
month
2012-01    1559
2012-02    1629
2012-03    2360
2012-04    2155
2012-05    2323
2012-06    1993
2012-07    2179
2012-08    2075
2012-09    1760
2012-10    1937
2012-11    1735
2012-12    1493
2013-01    1633
2013-02     886
2013-03    1306
2013-04    1692
2013-05    1605
2013-06    1317
2013-07    1484
2013-08    1354
2013-09    1212
2013-10    1391
2013-11    1209
2013-12    1008
2014-01    1098
2014-02     959
2014-03    1444
2014-04    1557
2014-05    1447
2014-06    1223
2014-07    1352
2014-08    1329
2014-09    1472
2014-10    1560
2014-11    1351
2014-12    1304
2015-01    1250
2015-02    1151
2015-03    1348
2015-04    1630
2015-05    1567
2015-06    1706
2015-07    1551
2015-08    1449
2015-09    1498
2015-10    1748
2015-11    1470
2015-12    1412
2016-01    1280
2016-02    1200
2016-03    1655
2016-04    1824
2016-05    1829
2016-06    1827
2016-07    1572
2016-08    1882
2016-09    1668
2016-10    1678
2016-11    1580
2016-12    1378
Name: 

---
## 3. Data Validation & Cleaning
Apply validation rules to ensure data quality. Records that fail validation are moved to the "failed" dataset.

In [10]:
# Initialize tracking columns for validation
df_clean = df_master.copy()
df_clean['_validation_failed'] = False
df_clean['_failure_reasons'] = ''

def flag_failed(df, mask, reason):
    """Flag records that fail a validation rule with the reason."""
    df.loc[mask, '_validation_failed'] = True
    df.loc[mask, '_failure_reasons'] = df.loc[mask, '_failure_reasons'].apply(
        lambda x: f"{x}; {reason}" if x else reason
    )
    count = mask.sum()
    if count > 0:
        print(f"  FAILED [{reason}]: {count} records")
    else:
        print(f"  PASSED [{reason}]: All records valid")
    return count

### 3.1 Validate Date (month column)

In [11]:
print("=" * 60)
print("VALIDATION: Date (month)")
print("=" * 60)

# Rule 1: month must match YYYY-MM format
valid_date_format = df_clean['month'].str.match(r'^\d{4}-(0[1-9]|1[0-2])$', na=False)
flag_failed(df_clean, ~valid_date_format, 'Invalid date format (expected YYYY-MM)')

# Rule 2: month must be within the target range
out_of_range = (df_clean['month'] < START_DATE) | (df_clean['month'] > END_DATE)
flag_failed(df_clean, out_of_range, f'Date outside range {START_DATE} to {END_DATE}')

# Rule 3: month must not be null
flag_failed(df_clean, df_clean['month'].isna(), 'Null date')

VALIDATION: Date (month)
  PASSED [Invalid date format (expected YYYY-MM)]: All records valid
  PASSED [Date outside range 2012-01 to 2016-12]: All records valid
  PASSED [Null date]: All records valid


np.int64(0)

### 3.2 Validate Town

In [12]:
print("=" * 60)
print("VALIDATION: Town")
print("=" * 60)

# Derive valid towns from the dataset's statistical properties
# We use towns that appear in the dataset as the reference list
# (since HDB towns are a known fixed set)
VALID_TOWNS = sorted(df_clean['town'].dropna().str.upper().str.strip().unique())
print(f"Valid towns derived from dataset ({len(VALID_TOWNS)}): {VALID_TOWNS}")

# Standardise town to uppercase
df_clean['town'] = df_clean['town'].str.upper().str.strip()

# Rule 1: town must not be null
flag_failed(df_clean, df_clean['town'].isna(), 'Null town')

# Rule 2: town must be in the valid set
invalid_town = ~df_clean['town'].isin(VALID_TOWNS) & df_clean['town'].notna()
flag_failed(df_clean, invalid_town, 'Town not in valid set')

# Rule 3: town must only contain letters and spaces
invalid_town_chars = ~df_clean['town'].str.match(r'^[A-Z\s/]+$', na=True) & df_clean['town'].notna()
flag_failed(df_clean, invalid_town_chars, 'Town contains invalid characters')

VALIDATION: Town


Valid towns derived from dataset (26): ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']
  PASSED [Null town]: All records valid
  PASSED [Town not in valid set]: All records valid
  PASSED [Town contains invalid characters]: All records valid


np.int64(0)

### 3.3 Validate Flat Type

In [13]:
print("=" * 60)
print("VALIDATION: Flat Type")
print("=" * 60)

# Derive valid flat types from the dataset
df_clean['flat_type'] = df_clean['flat_type'].str.upper().str.strip()
VALID_FLAT_TYPES = sorted(df_clean['flat_type'].dropna().unique())
print(f"Valid flat types ({len(VALID_FLAT_TYPES)}): {VALID_FLAT_TYPES}")

# Rule 1: flat_type must not be null
flag_failed(df_clean, df_clean['flat_type'].isna(), 'Null flat_type')

# Rule 2: flat_type must be in the valid set
invalid_flat_type = ~df_clean['flat_type'].isin(VALID_FLAT_TYPES) & df_clean['flat_type'].notna()
flag_failed(df_clean, invalid_flat_type, 'flat_type not in valid set')

# Rule 3: flat_type must follow the expected pattern (N ROOM or EXECUTIVE or MULTI-GENERATION)
valid_pattern = df_clean['flat_type'].str.match(
    r'^(1|2|3|4|5)\s+ROOM$|^EXECUTIVE$|^MULTI-GENERATION$', na=True
)
flag_failed(df_clean, ~valid_pattern & df_clean['flat_type'].notna(), 'flat_type unexpected format')

VALIDATION: Flat Type


Valid flat types (7): ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']
  PASSED [Null flat_type]: All records valid
  PASSED [flat_type not in valid set]: All records valid


  PASSED [flat_type unexpected format]: All records valid


np.int64(0)

### 3.4 Validate Flat Model

In [14]:
print("=" * 60)
print("VALIDATION: Flat Model")
print("=" * 60)

# Standardise flat_model to uppercase for consistency
df_clean['flat_model'] = df_clean['flat_model'].str.upper().str.strip()

VALID_FLAT_MODELS = sorted(df_clean['flat_model'].dropna().unique())
print(f"Valid flat models ({len(VALID_FLAT_MODELS)}): {VALID_FLAT_MODELS}")

# Rule 1: flat_model must not be null
flag_failed(df_clean, df_clean['flat_model'].isna(), 'Null flat_model')

# Rule 2: flat_model must be in the valid set
invalid_flat_model = ~df_clean['flat_model'].isin(VALID_FLAT_MODELS) & df_clean['flat_model'].notna()
flag_failed(df_clean, invalid_flat_model, 'flat_model not in valid set')

VALIDATION: Flat Model
Valid flat models (20): ['2-ROOM', 'ADJOINED FLAT', 'APARTMENT', 'DBSS', 'IMPROVED', 'IMPROVED-MAISONETTE', 'MAISONETTE', 'MODEL A', 'MODEL A-MAISONETTE', 'MODEL A2', 'MULTI GENERATION', 'NEW GENERATION', 'PREMIUM APARTMENT', 'PREMIUM APARTMENT LOFT', 'PREMIUM MAISONETTE', 'SIMPLIFIED', 'STANDARD', 'TERRACE', 'TYPE S1', 'TYPE S2']
  PASSED [Null flat_model]: All records valid
  PASSED [flat_model not in valid set]: All records valid


np.int64(0)

### 3.5 Validate Storey Range

In [15]:
print("=" * 60)
print("VALIDATION: Storey Range")
print("=" * 60)

df_clean['storey_range'] = df_clean['storey_range'].str.upper().str.strip()

VALID_STOREY_RANGES = sorted(df_clean['storey_range'].dropna().unique())
print(f"Valid storey ranges ({len(VALID_STOREY_RANGES)}): {VALID_STOREY_RANGES}")

# Rule 1: storey_range must not be null
flag_failed(df_clean, df_clean['storey_range'].isna(), 'Null storey_range')

# Rule 2: storey_range must match pattern "XX TO YY"
valid_storey_pattern = df_clean['storey_range'].str.match(r'^\d{2}\s+TO\s+\d{2}$', na=True)
flag_failed(df_clean, ~valid_storey_pattern & df_clean['storey_range'].notna(),
            'storey_range invalid format (expected XX TO YY)')

# Rule 3: lower bound must be less than upper bound
def validate_storey_bounds(val):
    """Check that lower storey <= upper storey."""
    if pd.isna(val):
        return True
    parts = str(val).split('TO')
    if len(parts) != 2:
        return True  # format issue already caught
    try:
        lower, upper = int(parts[0].strip()), int(parts[1].strip())
        return lower < upper and lower >= 1
    except ValueError:
        return True  # format issue already caught

invalid_bounds = ~df_clean['storey_range'].apply(validate_storey_bounds)
flag_failed(df_clean, invalid_bounds, 'storey_range lower >= upper or lower < 1')

VALIDATION: Storey Range
Valid storey ranges (25): ['01 TO 03', '01 TO 05', '04 TO 06', '06 TO 10', '07 TO 09', '10 TO 12', '11 TO 15', '13 TO 15', '16 TO 18', '16 TO 20', '19 TO 21', '21 TO 25', '22 TO 24', '25 TO 27', '26 TO 30', '28 TO 30', '31 TO 33', '31 TO 35', '34 TO 36', '36 TO 40', '37 TO 39', '40 TO 42', '43 TO 45', '46 TO 48', '49 TO 51']
  PASSED [Null storey_range]: All records valid


  PASSED [storey_range invalid format (expected XX TO YY)]: All records valid


  PASSED [storey_range lower >= upper or lower < 1]: All records valid


np.int64(0)

### 3.6 Additional Validation Rules

In [16]:
print("=" * 60)
print("ADDITIONAL VALIDATION RULES")
print("=" * 60)

# Rule: resale_price must be positive
df_clean['resale_price'] = pd.to_numeric(df_clean['resale_price'], errors='coerce')
flag_failed(df_clean, df_clean['resale_price'].isna(), 'Non-numeric resale_price')
flag_failed(df_clean, df_clean['resale_price'] <= 0, 'resale_price <= 0')

# Rule: floor_area_sqm must be positive
df_clean['floor_area_sqm'] = pd.to_numeric(df_clean['floor_area_sqm'], errors='coerce')
flag_failed(df_clean, df_clean['floor_area_sqm'].isna(), 'Non-numeric floor_area_sqm')
flag_failed(df_clean, df_clean['floor_area_sqm'] <= 0, 'floor_area_sqm <= 0')

# Rule: lease_commence_date must be a valid year and reasonable
df_clean['lease_commence_date'] = pd.to_numeric(df_clean['lease_commence_date'], errors='coerce')
flag_failed(df_clean, df_clean['lease_commence_date'].isna(), 'Non-numeric lease_commence_date')
flag_failed(df_clean, 
            (df_clean['lease_commence_date'] < 1930) | (df_clean['lease_commence_date'] > 2020),
            'lease_commence_date outside reasonable range (1930-2020)')

# Rule: block must not be null
flag_failed(df_clean, df_clean['block'].isna(), 'Null block')

# Rule: street_name must not be null
flag_failed(df_clean, df_clean['street_name'].isna(), 'Null street_name')

print(f"\nTotal records flagged for validation failure: {df_clean['_validation_failed'].sum()}")

ADDITIONAL VALIDATION RULES
  PASSED [Non-numeric resale_price]: All records valid
  PASSED [resale_price <= 0]: All records valid
  PASSED [Non-numeric floor_area_sqm]: All records valid
  PASSED [floor_area_sqm <= 0]: All records valid
  PASSED [Non-numeric lease_commence_date]: All records valid
  PASSED [lease_commence_date outside reasonable range (1930-2020)]: All records valid
  PASSED [Null block]: All records valid


  PASSED [Null street_name]: All records valid

Total records flagged for validation failure: 0


### 3.7 Recompute Remaining Lease
Assume HDB lease is 99 years. Remaining lease = (lease_commence_date + 99) - today, rounded down to years and months.

In [17]:
print("=" * 60)
print("RECOMPUTING REMAINING LEASE")
print("=" * 60)

def compute_remaining_lease(lease_commence_year):
    """Compute remaining lease as of today, given a 99-year lease starting Jan 1 of the commence year.
    
    Returns a string in 'XX years YY months' format, rounded down.
    Returns None if lease has expired or input is invalid.
    """
    if pd.isna(lease_commence_year):
        return None
    
    lease_commence_year = int(lease_commence_year)
    lease_end_year = lease_commence_year + HDB_LEASE_YEARS
    # Lease starts Jan 1 of commence year, ends Dec 31 of (commence + 99 - 1)
    # i.e., lease_end_date is Jan 1 of (commence + 99)
    lease_end_month = 1
    
    # Calculate difference: lease_end - today, rounded down
    total_months = (lease_end_year - TODAY.year) * 12 + (lease_end_month - TODAY.month)
    
    if total_months <= 0:
        return '0 years 00 months'
    
    years = total_months // 12
    months = total_months % 12
    return f"{years} years {months:02d} months"

df_clean['remaining_lease'] = df_clean['lease_commence_date'].apply(compute_remaining_lease)

print("Sample remaining lease values:")
print(df_clean[['lease_commence_date', 'remaining_lease']].drop_duplicates().sort_values(
    'lease_commence_date').head(20))

RECOMPUTING REMAINING LEASE


Sample remaining lease values:
        lease_commence_date     remaining_lease
390266                 1966  38 years 10 months
366747                 1967  39 years 10 months
367728                 1968  40 years 10 months
366744                 1969  41 years 10 months
367153                 1970  42 years 10 months
366743                 1971  43 years 10 months
366976                 1972  44 years 10 months
366642                 1973  45 years 10 months
366550                 1974  46 years 10 months
366726                 1975  47 years 10 months
366470                 1976  48 years 10 months
366493                 1977  49 years 10 months
366464                 1978  50 years 10 months
366463                 1979  51 years 10 months
366468                 1980  52 years 10 months
366471                 1981  53 years 10 months
366516                 1982  54 years 10 months
366597                 1983  55 years 10 months
366475                 1984  56 years 10 months
366492   

### 3.8 Handle Duplicate Keys
Composite key = all columns except resale_price. If duplicates exist, keep the higher price and discard the lower.

In [18]:
print("=" * 60)
print("DUPLICATE KEY HANDLING")
print("=" * 60)

# Define composite key columns (all columns except resale_price and internal tracking columns)
internal_cols = ['_validation_failed', '_failure_reasons', '_source_file']
key_cols = [c for c in df_clean.columns if c not in ['resale_price'] + internal_cols]
print(f"Composite key columns: {key_cols}")

# Identify duplicates
df_clean = df_clean.sort_values(by=key_cols + ['resale_price'], ascending=[True]*len(key_cols) + [False])
duplicate_mask = df_clean.duplicated(subset=key_cols, keep='first')

n_duplicates = duplicate_mask.sum()
print(f"Duplicate key records found: {n_duplicates}")

# Flag duplicates (lower-priced ones) for removal
flag_failed(df_clean, duplicate_mask, 'Duplicate key (lower resale_price discarded)')

if n_duplicates > 0:
    print("\nSample duplicate records:")
    dup_keys = df_clean[duplicate_mask][key_cols].head(3)
    for _, row in dup_keys.iterrows():
        match_mask = True
        for col in key_cols:
            match_mask = match_mask & (df_clean[col] == row[col])
        print(df_clean[match_mask][key_cols + ['resale_price']].to_string())
        print()

DUPLICATE KEY HANDLING
Composite key columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease']


Duplicate key records found: 1600
  FAILED [Duplicate key (lower resale_price discarded)]: 1600 records

Sample duplicate records:
          month        town flat_type block       street_name storey_range  floor_area_sqm      flat_model  lease_commence_date     remaining_lease  resale_price
366493  2012-01  ANG MO KIO    3 ROOM   256  ANG MO KIO AVE 4     10 TO 12            73.0  NEW GENERATION                 1977  49 years 10 months      353000.0
366494  2012-01  ANG MO KIO    3 ROOM   256  ANG MO KIO AVE 4     10 TO 12            73.0  NEW GENERATION                 1977  49 years 10 months      340000.0

          month        town flat_type block       street_name storey_range  floor_area_sqm      flat_model  lease_commence_date     remaining_lease  resale_price
366507  2012-01  ANG MO KIO    3 ROOM   506  ANG MO KIO AVE 8     07 TO 09            68.0  NEW GENERATION                 1980  52 years 10 months      342000.0
366508  2012-01  ANG MO KIO    3 ROOM   506  ANG MO KIO AV

          month        town flat_type block        street_name storey_range  floor_area_sqm      flat_model  lease_commence_date     remaining_lease  resale_price
366487  2012-01  ANG MO KIO    3 ROOM   558  ANG MO KIO AVE 10     10 TO 12            67.0  NEW GENERATION                 1980  52 years 10 months      344000.0
366478  2012-01  ANG MO KIO    3 ROOM   558  ANG MO KIO AVE 10     10 TO 12            67.0  NEW GENERATION                 1980  52 years 10 months      336000.0



### 3.9 Anomalous Resale Price Detection

**Heuristic 1 - IQR Method by Group**: For each (town, flat_type, storey_range), prices outside Q1 - 1.5*IQR and Q3 + 1.5*IQR are flagged.

**Heuristic 2 - Price-per-SQM Z-Score**: Compute price/sqm, then z-scores within (town, flat_type). Records with |z-score| > 3 are flagged.

These heuristics account for the natural variation in prices across different locations, flat types, and floors.

In [19]:
print("=" * 60)
print("ANOMALOUS RESALE PRICE DETECTION")
print("=" * 60)

# Only apply anomaly detection on records not already failed
valid_mask = ~df_clean['_validation_failed']

# Heuristic 1: IQR method by (town, flat_type, storey_range)
print("\n--- Heuristic 1: IQR Method ---")
group_cols_iqr = ['town', 'flat_type', 'storey_range']

def compute_iqr_bounds(group):
    """Return lower and upper IQR bounds for a group."""
    q1 = group['resale_price'].quantile(0.25)
    q3 = group['resale_price'].quantile(0.75)
    iqr = q3 - q1
    return pd.Series({
        'iqr_lower': q1 - 1.5 * iqr,
        'iqr_upper': q3 + 1.5 * iqr
    })

# Compute bounds for groups with enough data (min 5 records)
group_sizes = df_clean[valid_mask].groupby(group_cols_iqr).size()
sufficient_groups = group_sizes[group_sizes >= 5].index

iqr_bounds = df_clean[valid_mask].groupby(group_cols_iqr).apply(compute_iqr_bounds).reset_index()
df_clean = df_clean.merge(iqr_bounds, on=group_cols_iqr, how='left')

iqr_anomaly = (
    valid_mask & 
    df_clean['iqr_lower'].notna() &
    ((df_clean['resale_price'] < df_clean['iqr_lower']) | 
     (df_clean['resale_price'] > df_clean['iqr_upper']))
)
flag_failed(df_clean, iqr_anomaly, 'Anomalous price (IQR outlier)')
df_clean.drop(columns=['iqr_lower', 'iqr_upper'], inplace=True)

# Heuristic 2: Price-per-SQM Z-Score
print("\n--- Heuristic 2: Price-per-SQM Z-Score ---")
valid_mask = ~df_clean['_validation_failed']  # Refresh after IQR flagging
df_clean['_price_per_sqm'] = df_clean['resale_price'] / df_clean['floor_area_sqm']

group_cols_zscore = ['town', 'flat_type']
zscore_stats = df_clean[valid_mask].groupby(group_cols_zscore)['_price_per_sqm'].agg(['mean', 'std']).reset_index()
zscore_stats.columns = group_cols_zscore + ['_ppsqm_mean', '_ppsqm_std']

df_clean = df_clean.merge(zscore_stats, on=group_cols_zscore, how='left')
df_clean['_zscore'] = (df_clean['_price_per_sqm'] - df_clean['_ppsqm_mean']) / df_clean['_ppsqm_std']

zscore_anomaly = (
    valid_mask &
    df_clean['_zscore'].notna() &
    (df_clean['_zscore'].abs() > 3)
)
flag_failed(df_clean, zscore_anomaly, 'Anomalous price (z-score > 3 on price/sqm)')

# Clean up temp columns
df_clean.drop(columns=['_price_per_sqm', '_ppsqm_mean', '_ppsqm_std', '_zscore'], inplace=True)

print(f"\nTotal anomalous records flagged: {iqr_anomaly.sum() + zscore_anomaly.sum()}")

ANOMALOUS RESALE PRICE DETECTION

--- Heuristic 1: IQR Method ---


  PASSED [Anomalous price (IQR outlier)]: All records valid

--- Heuristic 2: Price-per-SQM Z-Score ---
  FAILED [Anomalous price (z-score > 3 on price/sqm)]: 492 records

Total anomalous records flagged: 492


### 3.10 Split into Cleaned and Failed Datasets

In [20]:
print("=" * 60)
print("SPLITTING CLEANED AND FAILED DATASETS")
print("=" * 60)

# Output columns (exclude internal tracking columns)
output_cols = [c for c in df_clean.columns if not c.startswith('_')]

# Split
df_passed = df_clean[~df_clean['_validation_failed']][output_cols].copy()
df_failed = df_clean[df_clean['_validation_failed']][output_cols + ['_failure_reasons']].copy()
df_failed.rename(columns={'_failure_reasons': 'failure_reasons'}, inplace=True)

print(f"Cleaned records: {len(df_passed)}")
print(f"Failed records:  {len(df_failed)}")
print(f"Total:           {len(df_passed) + len(df_failed)} (original: {len(df_clean)})")

if len(df_failed) > 0:
    print(f"\nFailure reasons breakdown:")
    # Count each individual reason
    all_reasons = df_failed['failure_reasons'].str.split('; ').explode()
    print(all_reasons.value_counts())

# Save cleaned data
cleaned_output_path = OUTPUT_DIR / 'cleaned_data.csv'
df_passed.to_csv(cleaned_output_path, index=False)
print(f"\nCleaned data saved to: {cleaned_output_path}")

# Save failed data
failed_output_path = OUTPUT_DIR / 'failed_data.csv'
df_failed.to_csv(failed_output_path, index=False)
print(f"Failed data saved to: {failed_output_path}")

SPLITTING CLEANED AND FAILED DATASETS
Cleaned records: 90452
Failed records:  2092
Total:           92544 (original: 92544)

Failure reasons breakdown:
failure_reasons
Duplicate key (lower resale_price discarded)    1600
Anomalous price (z-score > 3 on price/sqm)       492
Name: count, dtype: int64



Cleaned data saved to: output\cleaned_data.csv
Failed data saved to: output\failed_data.csv


---
## 4. Data Transformation
Create the Resale Identifier and apply hashing.

### 4.1 Create Resale Identifier

The Resale Identifier is composed of:
1. First character: "S"
2. Next 3 digits: first 3 digits of block (after removing non-digit characters), left-padded with zeros
3. Next 2 digits: 1st and 2nd digit of the average resale price grouped by (year-month, town, flat_type)
4. Last 2 digits: month number from the entry (e.g., 2012-01 -> 01)
5. Final character: first character of town

In [21]:
print("=" * 60)
print("CREATING RESALE IDENTIFIER")
print("=" * 60)

df_transformed = df_passed.copy()

# Component 1: "S" prefix (constant)

# Component 2: First 3 digits of block, removing non-digit chars, left-padded with zeros
def extract_block_digits(block_val):
    """Extract first 3 digits from block, removing any non-digit characters, left-pad with zeros."""
    digits = ''.join(c for c in str(block_val) if c.isdigit())
    return digits[:3].zfill(3)  # Take first 3 digits, pad if less than 3

df_transformed['_block_digits'] = df_transformed['block'].apply(extract_block_digits)

# Component 3: First 2 digits of average resale price by (month, town, flat_type)
avg_price = df_transformed.groupby(['month', 'town', 'flat_type'])['resale_price'].transform('mean')
df_transformed['_avg_price_digits'] = avg_price.apply(lambda x: str(int(x))[:2])

# Component 4: Month number from the entry (last 2 chars of month column)
df_transformed['_month_digits'] = df_transformed['month'].str[-2:]

# Component 5: First character of town
df_transformed['_town_char'] = df_transformed['town'].str[0]

# Assemble the Resale Identifier
df_transformed['resale_identifier'] = (
    'S' +
    df_transformed['_block_digits'] +
    df_transformed['_avg_price_digits'] +
    df_transformed['_month_digits'] +
    df_transformed['_town_char']
)

# Clean up temp columns
df_transformed.drop(columns=['_block_digits', '_avg_price_digits', '_month_digits', '_town_char'], inplace=True)

print("Sample Resale Identifiers:")
print(df_transformed[['month', 'town', 'flat_type', 'block', 'resale_price', 'resale_identifier']].head(10))
print(f"\nTotal unique identifiers: {df_transformed['resale_identifier'].nunique()} out of {len(df_transformed)} records")

CREATING RESALE IDENTIFIER
Sample Resale Identifiers:
     month        town flat_type block  resale_price resale_identifier
0  2012-01  ANG MO KIO    2 ROOM   170      260000.0         S1702501A
1  2012-01  ANG MO KIO    2 ROOM   174      226000.0         S1742501A
2  2012-01  ANG MO KIO    2 ROOM   314      263000.0         S3142501A
3  2012-01  ANG MO KIO    2 ROOM   314      275000.0         S3142501A
4  2012-01  ANG MO KIO    2 ROOM   406      257800.0         S4062501A
5  2012-01  ANG MO KIO    2 ROOM   508      260000.0         S5082501A
6  2012-01  ANG MO KIO    3 ROOM   118      320000.0         S1183401A
7  2012-01  ANG MO KIO    3 ROOM   121      382800.0         S1213401A
8  2012-01  ANG MO KIO    3 ROOM   151      302000.0         S1513401A
9  2012-01  ANG MO KIO    3 ROOM   154      321000.0         S1543401A

Total unique identifiers: 76903 out of 90452 records


### 4.2 Handle Duplicates After Transformation
If any duplicate records remain after adding the Resale Identifier, keep the higher-priced one.

In [22]:
print("=" * 60)
print("POST-TRANSFORMATION DUPLICATE HANDLING")
print("=" * 60)

# Check for full row duplicates (all columns except resale_price)
transform_key_cols = [c for c in df_transformed.columns if c != 'resale_price']
df_transformed = df_transformed.sort_values(
    by=transform_key_cols + ['resale_price'],
    ascending=[True] * len(transform_key_cols) + [False]
)

dup_mask_transform = df_transformed.duplicated(subset=transform_key_cols, keep='first')
n_dups = dup_mask_transform.sum()
print(f"Post-transformation duplicate records: {n_dups}")

if n_dups > 0:
    # Move duplicates to failed dataset
    df_transform_dups = df_transformed[dup_mask_transform].copy()
    df_transform_dups['failure_reasons'] = 'Post-transformation duplicate (lower price discarded)'
    df_failed = pd.concat([df_failed, df_transform_dups], ignore_index=True)
    df_transformed = df_transformed[~dup_mask_transform].copy()
    print(f"Removed {n_dups} post-transformation duplicates")

# Save transformed data
transformed_output_path = OUTPUT_DIR / 'transformed_data.csv'
df_transformed.to_csv(transformed_output_path, index=False)
print(f"\nTransformed data saved to: {transformed_output_path} ({len(df_transformed)} rows)")

POST-TRANSFORMATION DUPLICATE HANDLING


Post-transformation duplicate records: 0



Transformed data saved to: output\transformed_data.csv (90452 rows)

### 4.3 Hash the Resale Identifier

**Algorithm: SHA-256**

SHA-256 is used because:
- **Irreversible**: It is a one-way cryptographic hash function. The original input cannot be recovered from the hash output.
- **Collision-resistant**: The probability of two different inputs producing the same hash is astronomically low (2^-256), effectively preserving uniqueness.
- **Deterministic**: The same input always produces the same 64-character hexadecimal output.
- **Industry standard**: Widely used and trusted for data integrity and pseudonymisation.

In [23]:
print("=" * 60)
print("HASHING RESALE IDENTIFIER (SHA-256)")
print("=" * 60)

def sha256_hash(value):
    """Apply SHA-256 hash to a string value."""
    return hashlib.sha256(str(value).encode('utf-8')).hexdigest()

df_hashed = df_transformed.copy()
df_hashed['hashed_resale_identifier'] = df_hashed['resale_identifier'].apply(sha256_hash)

# Verify uniqueness preservation
n_unique_original = df_hashed['resale_identifier'].nunique()
n_unique_hashed = df_hashed['hashed_resale_identifier'].nunique()
print(f"Unique original identifiers: {n_unique_original}")
print(f"Unique hashed identifiers:   {n_unique_hashed}")
print(f"Uniqueness preserved: {n_unique_original == n_unique_hashed}")

print("\nSample hashed identifiers:")
print(df_hashed[['resale_identifier', 'hashed_resale_identifier']].head(5))

# Save hashed data (cleaned data + hashed identifier)
hashed_output_path = OUTPUT_DIR / 'hashed_data.csv'
df_hashed.to_csv(hashed_output_path, index=False)
print(f"\nHashed data saved to: {hashed_output_path} ({len(df_hashed)} rows)")

HASHING RESALE IDENTIFIER (SHA-256)
Unique original identifiers: 76903
Unique hashed identifiers:   76903
Uniqueness preserved: True

Sample hashed identifiers:
  resale_identifier                           hashed_resale_identifier
0         S1702501A  7b76c2e5ad415d73d9ac4e74b364f51dc0b60f0db59b52...
1         S1742501A  0509ae6467b1e75c84fe77c74c7f8857f8926009d7b912...
2         S3142501A  df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...
3         S3142501A  df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...
4         S4062501A  e22d10c12612bb3c3d1ff735a675db47039477c6ce8dbe...



Hashed data saved to: output\hashed_data.csv (90452 rows)


### 4.4 Update Failed Dataset (final save)

In [24]:
# Re-save failed data with any post-transformation duplicates included
failed_output_path = OUTPUT_DIR / 'failed_data.csv'
df_failed.to_csv(failed_output_path, index=False)
print(f"Final failed data saved to: {failed_output_path} ({len(df_failed)} rows)")

if len(df_failed) > 0:
    print("\nFinal failure reasons breakdown:")
    all_reasons = df_failed['failure_reasons'].str.split('; ').explode()
    print(all_reasons.value_counts())

Final failed data saved to: output\failed_data.csv (2092 rows)

Final failure reasons breakdown:
failure_reasons
Duplicate key (lower resale_price discarded)    1600
Anomalous price (z-score > 3 on price/sqm)       492
Name: count, dtype: int64


---
## 5. Output Summary

In [25]:
print("=" * 60)
print("ETL PIPELINE OUTPUT SUMMARY")
print("=" * 60)

output_files = {
    'Raw':         ('raw_data.csv',         'Original records within date range, unmodified'),
    'Cleaned':     ('cleaned_data.csv',     'Records that passed all validation rules'),
    'Transformed': ('transformed_data.csv', 'Cleaned data with Resale Identifier column'),
    'Failed':      ('failed_data.csv',      'Rejected records with failure reasons'),
    'Hashed':      ('hashed_data.csv',      'Transformed data with SHA-256 hashed identifier'),
}

for label, (filename, description) in output_files.items():
    filepath = OUTPUT_DIR / filename
    if filepath.exists():
        row_count = len(pd.read_csv(filepath))
        file_size = filepath.stat().st_size / 1024  # KB
        print(f"  {label:15s} | {filename:25s} | {row_count:>7,} rows | {file_size:>8.1f} KB")
        print(f"  {'':15s} | {description}")
    else:
        print(f"  {label:15s} | {filename:25s} | FILE NOT FOUND")
    print()

print("=" * 60)
print("ETL Pipeline completed successfully.")
print("=" * 60)

ETL PIPELINE OUTPUT SUMMARY
  Raw             | raw_data.csv              |  92,544 rows |   7888.7 KB
                  | Original records within date range, unmodified

  Cleaned         | cleaned_data.csv          |  90,452 rows |   9160.6 KB
                  | Records that passed all validation rules



  Transformed     | transformed_data.csv      |  90,452 rows |  10043.9 KB
                  | Cleaned data with Resale Identifier column

  Failed          | failed_data.csv           |   2,092 rows |    300.9 KB
                  | Rejected records with failure reasons

  Hashed          | hashed_data.csv           |  90,452 rows |  15785.5 KB
                  | Transformed data with SHA-256 hashed identifier

ETL Pipeline completed successfully.
